In [1]:
from utils.random_forest_utils.rf_preprocessing_utils import WindowAlgPreprocessor
import joblib

rf_preprocessor = WindowAlgPreprocessor(sensors_path="../../../../data/ml/features_machine_and_movement_complete.csv", target_path="../../../../data/ml/targets_movement_complete.csv")
sensors_df, target_df = rf_preprocessor.read_data()
sensors_df = rf_preprocessor.feature_selection()
rf_preprocessor.normalize_angle()

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.317461,0.337050,0.626034,0.344543
1,2,0.022017,0.319825,0.363018,0.600159,0.372211
2,2,0.044033,0.319035,0.401244,0.560835,0.412939
3,2,0.066050,0.317768,0.438220,0.522651,0.452336
4,2,0.088067,0.320230,0.465094,0.495879,0.480970
...,...,...,...,...,...,...
14558,318,0.902686,0.641596,0.373375,0.680266,0.314326
14559,318,0.924703,0.640550,0.372634,0.680719,0.313536
14560,318,0.946720,0.640334,0.374073,0.679194,0.315070
14561,318,0.968736,0.641037,0.377992,0.675413,0.319245


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy

# Make sure 'Time_[s]' is a column
df = sensors_df.reset_index()  # if 'Time_[s]' was the index

def resample_experiment(group, n=46, metric='mean'):
    """
    Resample experiment into n equal time segments and compute a single statistical metric.
    
    Parameters:
    - group: DataFrame of one experiment
    - n: number of time segments
    - metric: metric to compute. Options:
        'mean', 'median', 'min', 'max', 'range', 'std', 'var', 'mad', 
        'rms', 'skew', 'kurtosis', 'energy', 'entropy', 'cv', 'iqr', 
        'p25', 'p75', 'trend_slope'
    """
    # Ensure time is sorted
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]']

    # Create n time bins
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    segment_stats = []

    # Process each time segment
    for i in range(n):
        segment = group[(time_col >= time_bins[i]) & (time_col < time_bins[i + 1])]
        if segment.empty:
            continue

        stats_dict = {
            'Time_[s]': time_bins[i],
            # 'Time_end': time_bins[i + 1],
            'Experiment_ID': group['Experiment_ID'].iloc[0]
        }

        for col in group.columns:
            if col in ['Time_[s]', 'Experiment_ID']:
                continue
            values = segment[col].values
            if len(values) == 0:
                continue

            # Compute only the selected metric
            if metric == 'mean':
                stats_dict[f'{col}_mean'] = np.mean(values)
            elif metric == 'median':
                stats_dict[f'{col}_median'] = np.median(values)
            elif metric == 'min':
                stats_dict[f'{col}_min'] = np.min(values)
            elif metric == 'max':
                stats_dict[f'{col}_max'] = np.max(values)
            elif metric == 'range':
                stats_dict[f'{col}_range'] = np.ptp(values)
            elif metric == 'std':
                stats_dict[f'{col}_std'] = np.std(values)
            elif metric == 'var':
                stats_dict[f'{col}_var'] = np.var(values)
            elif metric == 'mad':
                stats_dict[f'{col}_mad'] = np.mean(np.abs(values - np.mean(values)))
            elif metric == 'rms':
                stats_dict[f'{col}_rms'] = np.sqrt(np.mean(np.square(values)))
            elif metric == 'skew':
                stats_dict[f'{col}_skew'] = skew(values)
            elif metric == 'kurtosis':
                stats_dict[f'{col}_kurtosis'] = kurtosis(values)
            elif metric == 'energy':
                stats_dict[f'{col}_energy'] = np.sum(np.square(values))
            elif metric == 'entropy':
                stats_dict[f'{col}_entropy'] = entropy(np.abs(values) / np.sum(np.abs(values)) + 1e-12)
            elif metric == 'cv':
                stats_dict[f'{col}_cv'] = np.std(values) / (np.mean(values) + 1e-12)
            elif metric == 'iqr':
                stats_dict[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            elif metric == 'p25':
                stats_dict[f'{col}_p25'] = np.percentile(values, 25)
            elif metric == 'p75':
                stats_dict[f'{col}_p75'] = np.percentile(values, 75)
            elif metric == 'trend_slope':
                stats_dict[f'{col}_trend_slope'] = np.polyfit(np.arange(len(values)), values, deg=1)[0] if len(values) > 1 else 0
            else:
                raise ValueError(f"Unknown metric: {metric}")

        segment_stats.append(stats_dict)

    return pd.DataFrame(segment_stats)

# Apply to each experiment
df_resampled = df.groupby('Experiment_ID').apply(lambda g: resample_experiment(g, n=46))
df_resampled = df_resampled.reset_index(drop=True)


/tmp/ipykernel_35184/1018919126.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_resampled = df.groupby('Experiment_ID').apply(lambda g: resample_experiment(g, n=46))


In [3]:
df_resampled

,Time_[s],Experiment_ID,MACHINE_BEND-DIE_LATERAL_Max_Torque_[%]_mean,MACHINE_BEND-DIE_ROTATING_Max_Torque_[%]_mean,MACHINE_BEND-DIE_VERTICAL_Max_Torque_[%]_mean,MACHINE_CLAMP-DIE_LATERAL_Max_Torque_[%]_mean,MACHINE_COLLET_AXIAL_Max_Torque_[%]_mean,MACHINE_MANDREL_AXIAL_Max_Torque_[%]_mean,MACHINE_PRESSURE-DIE_LATERAL_Max_Torque_[%]_mean,BEND-DIE_LATERAL_Movement_[mm]_mean,BEND-DIE_ROTATING_Angle_[°]_mean,CLAMP-DIE_LATERAL_Movement_[mm]_mean,COLLET_AXIAL_Movement_[mm]_mean,MANDREL_AXIAL_Movement_[mm]_mean,PRESSURE-DIE_AXIAL_Movement_[mm]_mean
0,0.000000,2,0.001188,0.998970,0.006242,0.000665,0.847964,0.997726,0.300843,0.668790,1.000000,0.000000,0.000009,9.999878e-01,0.205862
1,0.875000,2,0.001537,0.998935,0.003305,0.000448,0.847935,0.997821,0.067670,0.668790,1.000000,0.000000,0.000008,9.999878e-01,0.205862
2,1.750000,2,0.001188,0.999015,0.000000,0.000383,0.847281,0.997942,0.075345,0.668790,1.000000,0.000000,0.000008,9.999878e-01,0.205862
3,2.625000,2,0.001118,0.999155,0.000000,0.000363,0.845892,0.997821,0.083919,0.668790,1.000000,0.000000,0.000009,9.999878e-01,0.205862
4,3.500000,2,0.001188,0.998974,0.000624,0.000302,0.846406,0.997726,0.087979,0.668790,1.000000,0.000000,0.000009,9.999878e-01,0.205862
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14485,77.097826,318,0.073945,0.971258,0.263158,0.378432,0.893738,0.922329,0.785611,0.000004,0.441989,0.000002,0.999995,0.000000e+00,0.000037
14486,78.978261,318,0.074380,0.973478,0.273027,0.378444,0.893764,0.922329,0.785562,0.000004,0.575757,0.000002,0.999995,1.606179e-07,0.000037
14487,80.858696,318,0.074548,0.973728,0.265766,0.378455,0.892735,0.922577,0.785691,0.000004,0.707765,0.000002,0.999995,0.000000e+00,0.000037
14488,82.739130,318,0.074435,0.979728,0.252193,0.378461,0.893063,0.922329,0.785769,0.000004,0.839773,0.000002,0.999995,0.000000e+00,0.000037


In [4]:
def normalize_experiment(group, n=46):
    if len(group) > n:
        # Just take the first 46 rows
        return group.iloc[:n].copy()
    else:
        # Already 46 rows
        return group.copy()

# Apply to each experiment
df_normalized = target_df.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
df_normalized = df_normalized.reset_index(drop=True)


/tmp/ipykernel_35184/1968673034.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized = target_df.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)


In [5]:
count= 2
X = rf_preprocessor.group_and_pad(df_resampled, group_col="Experiment_ID")[:50,:,count:]
Y = rf_preprocessor.group_and_pad(df_normalized, group_col="Experiment_ID")[:50,:-1,1:]

In [6]:
import torch
import torch.nn as nn

x_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(Y, dtype=torch.float32)

feature_names = list(sensors_df.columns)[count:]

# --------------------------
# More Complex LSTM Model
# --------------------------
class ComplexLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers, 
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_dim*2)
        self.fc1 = nn.Linear(hidden_dim*2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        out, _ = self.lstm(x)                # (batch, seq_len, hidden_dim*2)
        out = self.layer_norm(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out


In [7]:
model = ComplexLSTM(input_dim=X.shape[-1], hidden_dim=128, output_dim=Y.shape[-1], num_layers=1)

/home/arman/Documents/university/Master/Master these 2025/tube-geometry-prediction/.venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


In [8]:
# '''
from torch import optim
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)

# --------------------------
# Training loop
# --------------------------
epochs = 200
batch_size = 32

for epoch in range(epochs):
    permutation = torch.randperm(x_tensor.size(0))
    epoch_loss = 0
    
    for i in range(0, x_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = x_tensor[indices], y_tensor[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.6f}")
    

model_path = "../../../../models/xai/LSTM.joblib"
joblib.dump(model, model_path)
print("Training complete!")
# '''

Epoch 1/200, Loss: 0.293031
Epoch 2/200, Loss: 0.280389
Epoch 3/200, Loss: 0.267028
Epoch 4/200, Loss: 0.257400
Epoch 5/200, Loss: 0.243752
Epoch 6/200, Loss: 0.233316
Epoch 7/200, Loss: 0.221922
Epoch 8/200, Loss: 0.211908
Epoch 9/200, Loss: 0.201875
Epoch 10/200, Loss: 0.190351
Epoch 11/200, Loss: 0.183992
Epoch 12/200, Loss: 0.176011
Epoch 13/200, Loss: 0.163616
Epoch 14/200, Loss: 0.157292
Epoch 15/200, Loss: 0.148113
Epoch 16/200, Loss: 0.140298
Epoch 17/200, Loss: 0.132478
Epoch 18/200, Loss: 0.127291
Epoch 19/200, Loss: 0.118240
Epoch 20/200, Loss: 0.114252
Epoch 21/200, Loss: 0.105678
Epoch 22/200, Loss: 0.099611
Epoch 23/200, Loss: 0.093192
Epoch 24/200, Loss: 0.088663
Epoch 25/200, Loss: 0.082285
Epoch 26/200, Loss: 0.079427
Epoch 27/200, Loss: 0.073347
Epoch 28/200, Loss: 0.069244
Epoch 29/200, Loss: 0.063546
Epoch 30/200, Loss: 0.060064
Epoch 31/200, Loss: 0.058201
Epoch 32/200, Loss: 0.052690
Epoch 33/200, Loss: 0.049655
Epoch 34/200, Loss: 0.045424
Epoch 35/200, Loss: 0.0

In [9]:
model = joblib.load('../../../../models/xai/LSTM.joblib')

In [11]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from captum.attr import GuidedBackprop, DeepLift, IntegratedGradients, Saliency, InputXGradient, FeatureAblation, KernelShap, GradientShap
import torch.nn as nn

# --------------------------
# Attribution computation
# --------------------------
def compute_attr(sample_input, output_feature, time_step, method_cls):
    """Compute attributions for a specific output feature and time step."""
    class WrappedModel(nn.Module):
        def __init__(self, base_model, time_step, output_feature):
            super().__init__()
            self.base_model = base_model
            self.time_step = time_step
            self.output_feature = output_feature

        def forward(self, x):
            out = self.base_model(x)
            return out[:, self.time_step, self.output_feature]

    wrapped_model = WrappedModel(model, time_step, output_feature)
    method = method_cls(wrapped_model)
    
    # Handle different method requirements
    method_name = method_cls.__name__
    
    if method_name in ["GradientShap", "IntegratedGradients", "DeepLift", "DeepLiftShap"]:
        # These methods typically require baselines
        baselines = torch.zeros_like(sample_input)
        attr = method.attribute(sample_input, baselines=baselines)
    elif method_name in ["Occlusion", "FeatureAblation"]:
        # Perturbation-based methods may need different parameters
        attr = method.attribute(sample_input, 
                               sliding_window_shapes=(1, 10))  # Adjust for your data
    else:
        # Methods like Saliency, GuidedBackprop, InputXGradient
        attr = method.attribute(sample_input)
    
    return attr.detach().numpy().reshape(sample_input.shape[1], sample_input.shape[2])

# Map output features to names
n_target = 4
target = range(n_target)
target_features = target_df.columns.to_list()[-n_target:]
output_feature_dict = {f: t for f, t in zip(target, target_features)}

# --------------------------
# Widgets
# --------------------------
sample_selector = widgets.IntSlider(
    value=0, min=0, max=x_tensor.shape[0]-1, step=1, description='Sample'
)

feature_selector = widgets.IntSlider(
    value=0, min=0, max=y_tensor.shape[2]-1, step=1, description='Output Feature'
)

time_selector = widgets.IntSlider(
    value=0, min=0, max=x_tensor.shape[1]-1, step=1, description='Time Step'
)

methods = {
    "GuidedBackprop": GuidedBackprop,
    "DeepLift": DeepLift,
    "IntegratedGradients": IntegratedGradients,
    "Saliency": Saliency,
    "InputXGradient": InputXGradient,
    "FeatureAblation": FeatureAblation,
    "KernelSHAP": KernelShap,
    "GradientSHAP": GradientShap,
}


method_selector = widgets.Dropdown(
    options=list(methods.keys()),
    value=list(methods.keys())[0],
    description='Method'
)

out = widgets.Output()

# --------------------------
# Update plot function
# --------------------------
def update_plot(change):
    sample_idx = sample_selector.value
    output_feature_idx = feature_selector.value
    time_step = time_selector.value
    method_name = method_selector.value

    # Compute attribution
    sample_input = x_tensor[sample_idx:sample_idx+1]
    attr_np = compute_attr(sample_input, output_feature_idx, time_step, methods[method_name])

    # Extract model outputs
    model_output = model(sample_input).detach().numpy()[0]
    target_value = model_output[time_step, output_feature_idx]
    sensor_values = sample_input[0].detach().numpy()
    time_values = np.arange(sample_input.shape[1])  # actual time steps

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

        # 1. Attribution heatmap
        im = axes[0].imshow(attr_np.T, cmap='inferno', aspect='auto')
        axes[0].set_title(f"Attribution Map: {method_name}", fontsize=12)
        axes[0].set_ylabel("Feature")
        axes[0].set_xlabel("Time Step")
        axes[0].set_xticks(time_values)
        axes[0].set_yticks(np.arange(len(feature_names)))
        axes[0].set_yticklabels(feature_names, fontsize=9)
        cbar = fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=8)

        # 2. Model output vs true output
        axes[1].plot(time_values, model_output[:, output_feature_idx], label='Model Output', color='blue')
        axes[1].plot(time_values, Y[sample_idx, :, output_feature_idx], label='True Output', color='orange')
        axes[1].scatter(time_step, target_value, color='red', zorder=5, label='Selected Time Step')
        axes[1].set_ylabel("Target Value")
        axes[1].set_xlabel("Time Step")
        axes[1].set_xticks(time_values)
        axes[1].set_title(f"Model Output: {output_feature_dict[output_feature_idx]}", fontsize=12)
        axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=9)

        # 3. Sensor input values
        for i, fname in enumerate(feature_names):
            axes[2].plot(time_values, sensor_values[:, i], label=fname)
        axes[2].set_xlabel("Time Step")
        axes[2].set_ylabel("Sensor Value")
        axes[2].set_xticks(time_values)
        axes[2].set_title("Input Sensor Values", fontsize=12)
        axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=9)

        plt.tight_layout()
        plt.show()


# --------------------------
# Link widgets
# --------------------------
for w in [sample_selector, feature_selector, time_selector, method_selector]:
    w.observe(update_plot, names='value')

display(widgets.VBox([sample_selector, feature_selector, time_selector, method_selector, out]))

# Initial plot
update_plot(None)


/home/arman/Documents/university/Master/Master these 2025/tube-geometry-prediction/.venv/lib/python3.11/site-packages/captum/attr/_core/guided_backprop_deconvnet.py:63: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)
/home/arman/Documents/university/Master/Master these 2025/tube-geometry-prediction/.venv/lib/python3.11/site-packages/captum/attr/_core/guided_backprop_deconvnet.py:66: UserWarning: Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished
  warnings.warn(


# Feature Ablation:
- Remove one feature at a time (set it to a baseline value)
- Measure how much the model's output changes
- Important features will cause large output changes when removed

# Guided Backpropagation:
is a gradient-based attribution method that enhances traditional backpropagation to produce cleaner, more interpretable feature visualizations.
- Modifies the standard backpropagation process
- Zeros out negative gradients during backpropagation
- Combines ideas from DeconvNet and backpropagation
- Results in cleaner visualizations showing only "positive evidence"

# InputXGradient
is a gradient-based attribution method that multiplies the input features by their gradients to determine feature importance. It's an extension of the simpler Saliency method and provides more informative attributions by accounting for both the gradient signal and the input feature magnitud
- for a given input and model, it computes the gradient of the output with respect to the input, and then performs an element-wise multiplication of the input by this gradient

# SHAP Values: 
- The core idea is to treat the features of a data instance as players in a coalitional game. 
- The "payout" is the difference between the model's actual prediction and its average prediction. 
- SHAP values fairly distribute this payout among the features according to their marginal contribution across all possible feature combinations (coalitions). 
- They satisfy desirable properties like Efficiency (the SHAP values add up to the difference between the prediction and the average prediction) and Symmetry

# KernelSHAP: 
- This is a model-agnostic approximation method. 
- Since calculating exact Shapley values is computationally expensive, KernelSHAP uses a smart sampling approach. 
- It generates different "coalitions" of features (where some features are "on" and some are "off"), and then uses a weighted linear regression to estimate the Shapley values.
- The "kernel" refers to a specific weighting function that gives higher weight to coalitions with either very few or very many features

# GradientSHAP: 
- This method is designed for differentiable models like neural networks. 
- It combines ideas from Integrated Gradients, SHAP, and SmoothGrad. 
- It works by calculating the expected value of the gradients when the input is interpolated between a baseline (from a background dataset) and the actual instance to be explained. 
- This expectation is computed by sampling multiple baselines and adding noise to the input
- In practice, this is often faster than KernelSHAP for deep learning models.